# Mini-GPT on tinyshakespeare (Module 8)

**Setup:** Runtime → Change runtime type → **T4 GPU** → Save.

Run cells top-to-bottom. Total time ~5–10 min on T4.

In [ ]:
!nvidia-smi

In [ ]:
import os, math, time, urllib.request
import torch, torch.nn as nn, torch.nn.functional as F

URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
PATH = 'tinyshakespeare.txt'
if not os.path.exists(PATH):
    urllib.request.urlretrieve(URL, PATH)

text = open(PATH).read()
chars = sorted(set(text))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f'corpus chars: {len(text):,} | vocab: {V}')

## Model (same as Module 7's `gpt_solution.py`, inlined)

In [ ]:
class GPTConfig:
    vocab_size = V; block_size = 128
    n_layer = 6; n_head = 6; n_embd = 192; dropout = 0.2

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head; self.n_embd = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3*cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout); self.resid_drop = nn.Dropout(cfg.dropout)
        self.dropout = cfg.dropout
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v,
                dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd); self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.n_embd, 4*cfg.n_embd), nn.GELU(),
            nn.Linear(4*cfg.n_embd, cfg.n_embd), nn.Dropout(cfg.dropout))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks: x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

## Training

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
cfg = GPTConfig()
torch.manual_seed(1337)
model = GPT(cfg).to(device)
print(f'params: {sum(p.numel() for p in model.parameters())/1e6:.2f} M')
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1, betas=(0.9, 0.95))
scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))

def get_batch(split, batch_size=64):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - cfg.block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+cfg.block_size] for i in ix])
    y = torch.stack([d[i+1:i+cfg.block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(eval_iters=50):
    model.eval()
    out = {}
    for split in ('train', 'val'):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, l = model(x, y); losses[k] = l.item()
        out[split] = losses.mean().item()
    model.train()
    return out

t0 = time.time(); model.train()
for step in range(3000):
    if step % 500 == 0:
        L = estimate_loss()
        print(f'step {step:5d} | train {L["train"]:.4f} | val {L["val"]:.4f} | {time.time()-t0:.0f}s')
    x, y = get_batch('train')
    with torch.cuda.amp.autocast(enabled=(device=='cuda')):
        _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()
torch.save(model.state_dict(), 'ckpt.pt')

## Sample

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new=500, temperature=0.8, top_k=40):
    model.eval()
    for _ in range(max_new):
        idx_cond = idx[:, -cfg.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float('inf')
        probs = F.softmax(logits, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    return idx

seed = torch.zeros((1, 1), dtype=torch.long, device=device)
out = generate(model, seed)
print(decode(out[0].tolist()))